<a href="https://colab.research.google.com/github/yous92/TReND-CaMinA/blob/main/notebooks/Kenya26/03-04-AllenTutorial/project_templates/Project_2_BongoSpikes_Comparing_CellTypes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Project 2** Predicting Cell Types from Neuron Activity

## How well can you predict what cell type a neuron belongs to based on its tuning properties?

We already saw that different types of cells have different responses to the drifting grating stimulus. Are these differences enough that we can predict what type of cell a neuron is based on it's tuning curve?

We sampled 13 different transgenic lines in this dataset. Some of the key differences are: excitatory vs inhibitory cells; cells that are uniquely expressed in specific layers; and cells that have distinct projection patterns.

For this project we'll focus on the three different inhibitory cell types: Pvalb, VIP, and SST. These three interneurons (inhibitory cells) form a microcircuit in the cortex. We've already seen that their responses to the drifting grating stimulus can be pretty different. The question is, how accurately can you predict these cell types based on their responses.


This project is conceptually similar to Project 1. We will compute the 2D tuning curve for all the sessions of each cell type within V1, and then flatten those tuning curves, and then set up training and hold out data. If you are unfamiliar with the flattening process, look at the Project 1 template to understand how that works.

In [ ]:
# @title Run to initialize Allen Brain Observatory on Colab {display-mode: "form" }

# run only once per runtime/session, and only if running in colab
# the runtime will need to restxart after
%%capture
!apt install s3fs

!pip uninstall -y numpy pandas
!pip install git+https://github.com/AllenInstitute/AllenSDK@1bdca3ad884c3a5edea8236161424650603e6f29 "numpy == 1.26.4" "pandas == 2.3.0" "matplotlib > 3.8.0" "statsmodels >= 0.14.4"
import allensdk
print('allensdk imported successfully')

!mkdir -p /data/allen-brain-observatory/
!s3fs allen-brain-observatory /data/allen-brain-observatory/ -o public_bucket=1

import time
print("Runtime is now restarting...")
print("You can ignore the error message [Your session crashed for an unknown reason.]")
time.sleep(5)
exit()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
from allensdk.core.brain_observatory_cache import BrainObservatoryCache

In [ ]:
import platform, os, sys
platstring = platform.platform()

if ('amzn' in platstring) or ('google.colab' in sys.modules):
    # for AWS
    vc_cache_dir = '/data/allen-brain-observatory/visual-coding-2p'
else:
    # for local drive, different operating systems
    if ('Darwin' in platstring) or ('macOS' in platstring):
        # OS X
        data_root = "/Volumes/TReND2026/"
    elif 'Windows'  in platstring:
        # Windows (replace with the drive letter of USB drive)
        data_root = "E:/"
    else:
        # your own linux platform
        # EDIT location where you mounted hard drive
        data_root = "/media/$USERNAME/TReND2026/"

    # visual behavior cache directory
    vc_cache_dir = os.path.join(data_root, "allen-brain-observatory","visual-coding-2p")

boc = BrainObservatoryCache(manifest_file=os.path.join(vc_cache_dir, 'manifest.json'))

In [ ]:
def compute_tuning(session_id):
    """
    Computes the 2D tuning array for all neurons in a given session
    for drifting gratings stimulus.

    Parameters
    ----------
    session_id : int
        The session id for one session.

    Returns
    -------
    dff : numpy.ndarray
        Delta F over F traces for all cells.
    stim_table : pandas.DataFrame
        Stimulus table for drifting gratings.
    trial_response : pandas.DataFrame
        Response per trial for all neurons.
    tuning_array : numpy.ndarray
        2D tuning array for all neurons, with dimensions (orientation, temporal_frequency, cell index).
    """
    #access data for the session
    data_set = boc.get_ophys_experiment_data(session_id)

    #get the DFF trace for all cells
    timestamps, dff = data_set.get_dff_traces()
    number_cells = dff.shape[0]
    print("Number of cells: ", number_cells)

    #get the stimulus table for the drifting gratings
    stim_table = data_set.get_stimulus_table('drifting_gratings')

    #get the orivals and tfvals
    all_ori = np.unique(stim_table.orientation)
    orivals = all_ori[np.isfinite(all_ori)]
    tfvals = np.unique(stim_table.temporal_frequency)
    tfvals = tfvals[np.isfinite(tfvals)]

    #compute response per trial for all neurons
    trial_response = pd.DataFrame(index=stim_table.index.values, columns=np.array(range(number_cells)).astype(str))

    for ind,row in stim_table.iterrows():
        for nc in range(number_cells):
            trial_response.loc[ind, str(nc)] = dff[nc,int(row.start):int(row.end)].mean()

    #calculate the response to the blanksweeps:
    blank_response = trial_response[np.isnan(stim_table.orientation)].mean()

    #compute 2D tuning array and subtract the blanksweep response
    tuning_array = np.empty((8,5,number_cells))
    for i,tf in enumerate(tfvals): #iterate across TF
        for j,ori in enumerate(orivals): #iterate across ori
            tuning_array[j,i,:] = trial_response[(stim_table.orientation==ori)&
                                                 (stim_table.temporal_frequency==tf)].mean() - blank_response

    return(dff, stim_table, trial_response, tuning_array)

Get a dataframe of all the experiments with Sst-IRES-Cre (SST) data in VISp for the session with the drifting grating stimulus

In [ ]:
exps_sst = pd.DataFrame(boc.get_ophys_experiments(targeted_structures=['VISp'],
                                              cre_lines=['Sst-IRES-Cre'],
                                             stimuli=['drifting_gratings']))

In [ ]:
len(exps_sst)

17

Get a dataframe of all the experiments with Vip-IRES-Cre (VIP) data in VISp for the session with the drifting grating stimulus

In [ ]:
exps_vip = pd.DataFrame(boc.get_ophys_experiments(targeted_structures=['VISp'],
                                              cre_lines=['Vip-IRES-Cre'],
                                             stimuli=['drifting_gratings']))

In [ ]:
len(exps_vip)

17

Get a dataframe of all the experiments with Pvalb-IRES-Cre (Pvalb or PV) data in VISp for the session with the drifting grating stimulus

In [ ]:
exps_pvalb = pd.DataFrame(boc.get_ophys_experiments(targeted_structures=['VISp'],
                                              cre_lines=['Pvalb-IRES-Cre'],
                                             stimuli=['drifting_gratings']))

In [ ]:
len(exps_pvalb)

16

Use the compute_tuning function to compute 2D tuning array for all SST sessions. Flatten the tuning array from the session, and append the result to a growing list of tuning data. (see Project 1 to understand the flattening)

In [ ]:
#create an empty array to contain the results
tuning_sst = np.empty((40,0))

#for each session
for pmid in exps_sst.id:
    print("Session ID: ", pmid)
    #compute the 2D tuning array
    (dff, stim_table, trial_response, tuning_array) = compute_tuning(pmid)
    #flatten the array from (8,5,numbercells) to (40,numbercells)
    ta_flatten = tuning_array.reshape(-1,tuning_array.shape[-1])
    #append the result to the array for results
    tuning_sst = np.append(tuning_sst, ta_flatten, axis=1)

Session ID:  581150104
Number of cells:  8
Session ID:  613968705
Number of cells:  15
Session ID:  576095926
Number of cells:  15
Session ID:  575939366
Number of cells:  5
Session ID:  577379202
Number of cells:  6
Session ID:  573720508
Number of cells:  10
Session ID:  612044635
Number of cells:  8
Session ID:  596779487
Number of cells:  11
Session ID:  582918858
Number of cells:  17
Session ID:  580163817
Number of cells:  7
Session ID:  590047029
Number of cells:  7
Session ID:  581026088
Number of cells:  15
Session ID:  580043440
Number of cells:  10
Session ID:  598635821
Number of cells:  14
Session ID:  584196534
Number of cells:  13
Session ID:  609894681
Number of cells:  7
Session ID:  601423209
Number of cells:  10


What is the shape of your resulting array?

In [ ]:
tuning_sst.shape

(40, 178)

Repeat this for the VIP data

In [ ]:
#create an empty array to contain the results
tuning_vip = np.empty((40,0))

#for each session
for pmid in exps_vip.id:
    print("Session ID: ", pmid)
    #compute the 2D tuning array
    (dff, stim_table, trial_response, tuning_array) = compute_tuning(pmid)
    #flatten the array from (8,5,numbercells) to (40,numbercells)
    ta_flatten = tuning_array.reshape(-1,tuning_array.shape[-1])
    #append the result to the array for results
    tuning_vip = np.append(tuning_vip, ta_flatten, axis=1)

Session ID:  663479824
Number of cells:  10
Session ID:  662361096
Number of cells:  16
Session ID:  657078119
Number of cells:  9
Session ID:  652091264
Number of cells:  17
Session ID:  657390171
Number of cells:  9
Session ID:  652842495
Number of cells:  4
Session ID:  657389972
Number of cells:  4
Session ID:  612543999
Number of cells:  13
Session ID:  658518486
Number of cells:  12
Session ID:  659495103
Number of cells:  15
Session ID:  657650110
Number of cells:  12
Session ID:  583279803
Number of cells:  11
Session ID:  585900296
Number of cells:  9
Session ID:  650079244
Number of cells:  24
Session ID:  606353987
Number of cells:  13
Session ID:  617395455
Number of cells:  13
Session ID:  657775947
Number of cells:  8


What is the shape of your resulting array?

In [ ]:
tuning_vip.shape

(40, 199)

Repeat this for the Pvalb data

In [ ]:
#create an empty array to contain the results
tuning_pvalb = np.empty((40,0))

#for each session
for pmid in exps_pvalb.id:
    print("Session ID: ", pmid)
    #compute the 2D tuning array
    (dff, stim_table, trial_response, tuning_array) = compute_tuning(pmid)
    #flatten the array from (8,5,numbercells) to (40,numbercells)
    ta_flatten = tuning_array.reshape(-1,tuning_array.shape[-1])
    #append the result to the array for results
    tuning_pvalb = np.append(tuning_pvalb, ta_flatten, axis=1)

Session ID:  673171528
Number of cells:  8
Session ID:  712178483
Number of cells:  6
Session ID:  671618887
Number of cells:  16
Session ID:  669861524
Number of cells:  10
Session ID:  680150733
Number of cells:  13
Session ID:  672206735
Number of cells:  9
Session ID:  676503588
Number of cells:  24
Session ID:  672211004
Number of cells:  11
Session ID:  710778377
Number of cells:  10
Session ID:  670395999
Number of cells:  3
Session ID:  671164733
Number of cells:  32
Session ID:  710504563
Number of cells:  17
Session ID:  675477919
Number of cells:  10
Session ID:  692345003
Number of cells:  7
Session ID:  673914981
Number of cells:  13
Session ID:  670728674
Number of cells:  18


What is the shape of your resulting array

In [ ]:
tuning_pvalb.shape

(40, 207)

Note: if you want to save these arrays, this would allow you to reload them without needing to recompute them in future session. Feel free to do that if you would like.



---

Now that we have our tuning arrays for these three different cell types, we want to create a labels for each neuron. We'll label SST as 0, VIP as 1, and Pvalb as 2.

In [ ]:
label = np.repeat(0,tuning_sst.shape[1]) #label 0 = SST
label = np.append(label, np.repeat(1,tuning_vip.shape[1])) #label 1 = VIP
label = np.append(label, np.repeat(2,tuning_pvalb.shape[1])) #label 2 = Pvalb

In [ ]:
label.shape

(584,)

Plot this to confirm it looks like you expect.

In [ ]:
plt.plot(label)

Now let's append the three tuning arrays together. Do SST first and then VIP and then Pvalb, to match the labels.



In [ ]:
tuning = np.append(tuning_sst, tuning_vip, axis=1)
tuning = np.append(tuning, tuning_pvalb, axis=1)

In [ ]:
tuning.shape

(40, 584)

Notice that our label vector is as long as our tuning array.

The challenge in this project is to use classification methods to determine the correct label for a neuron based on its tuning. You can try a variety of methods and determine which ones work best, and how well you can differentiate these two regions (it won't be perfect!).

You will want to set up some hold out data to test your classification methods. So let's do that now. We will create a random list of integers to remove from these arrays. First, set a seed so that we create the same random selection if we rerun this code. Set the seed variable with your favorite number.

In [ ]:
# Create a generator instance with a specific seed
rng = np.random.default_rng(seed=15)

Now get random numbers from the range of the number of neurons we have here. We'll select 80 neurons - but you can change this if you want to hold out more or less data. This serves as an index into your tuning array and label vector.

In [ ]:
total_cells = len(label)
holdout_length = 80
holdout = np.random.choice(range(total_cells), holdout_length, replace=False)

In [ ]:
len(np.where(label[holdout]==2)[0])

30

Use this to separate your tuning and label into a holdout set and a training set.

In [ ]:
tuning_holdout = tuning[:,holdout] #select the holdout indices
tuning_holdout.shape

(40, 80)

In [ ]:
tuning_train = np.delete(tuning, holdout, axis=1) #remove the holdout indices
tuning_train.shape

(40, 504)

In [ ]:
label_holdout = label[holdout] #select the holdout indices
label_holdout.shape

(80,)

In [ ]:
label_train = np.delete(label, holdout) #remove the holdout indices
label_train.shape

(504,)

From here, use your tuning_train and label_train to train classifier to best predict the brain region that each neuron is from. Sklearn has a number of classification methods that you can use and compare.


Other directions you can take this project in:

Here we used the tuning array for the response to drifting gratings to predict brain region. Could you use other aspects of the neurons' responses? For example, try using the dff traces during the spontaneous activity or the natural movie response. What other types of features could you look at?

1.   We see noticeable differences between the inhibitory neurons, but the differences between excitatory neurons are much more subtle. Nonetheless you can try to see how well you can distinguish these. A few comparisons that might be interesting:
  - layer 2/3 vs layer 5.
  

```
exps_23 = pd.DataFrame(boc.get_ophys_experiments(targeted_structures=['VISp'],
                                              cre_lines=['Cux2-CreERT2'],
                                              imaging_depths=[175],
                                             stimuli=['drifting_gratings']))
exps_5 = pd.DataFrame(boc.get_ophys_experiments(targeted_structures=['VISp'],
                                              cre_lines=['Rbp4-Cre_KL100'],
                                             stimuli=['drifting_gratings']))
```

  - layer 4 subtypes


```
exps_rorb = pd.DataFrame(boc.get_ophys_experiments(targeted_structures=['VISp'],
                                              cre_lines=['Rorb-IRES2-Cre'],
                                             stimuli=['drifting_gratings']))
exps_scnn = pd.DataFrame(boc.get_ophys_experiments(targeted_structures=['VISp'],
                                              cre_lines=['Scnn1a-Tg3-Cre'],
                                             stimuli=['drifting_gratings']))
exps_nr5a = pd.DataFrame(boc.get_ophys_experiments(targeted_structures=['VISp'],
                                              cre_lines=['Nr5a1-Cre'],
                                             stimuli=['drifting_gratings']))
```
How well can you differentiate these different cell types?



2.   We made our predictions from the tuning curves of these neurons. Can you try making predictions from other features, such as the DFF during spontaneous activity or during natural movies? Tuned responses to other stimuli? What other features could you try?

